# 02 Data Cleaning


The goal is to turn the raw Telco Customer Churn data into a clean, analysis-ready interim dataset while keeping the raw data unchanged.

## Cleaning Goals

From the exploration notebook, we know that the main cleaning tasks are:

- Load the raw dataset from `data/raw/Telco-Customer-Churn.csv`.
- Preserve the raw file unchanged.
- Convert `TotalCharges` from text to a numeric column.
- Handle blank `TotalCharges` values explicitly.
- Encode the target variable `Churn` as a binary numeric column.
- Check for duplicates and unexpected target values.
- Save the cleaned interim dataset to `data/interim/telco_churn_cleaned.csv`.

We keep the cleaning logic in this notebook for now. Refactoring into `src/` comes later as a separate learning step.

## Cleaning Decisions

These decisions are deliberately simple and visible:

| Issue | Decision | Reason |
| --- | --- | --- |
| `TotalCharges` is loaded as text | Convert with `pd.to_numeric(..., errors="coerce")` | Makes invalid values visible as missing values. |
| Blank `TotalCharges` values | Set to `0.0` only when `tenure == 0` | These are new customers with no accumulated charges yet. |
| `Churn` is `Yes`/`No` | Add `ChurnBinary`: `Yes -> 1`, `No -> 0` | Keeps the original business label and creates a model-ready target. |
| `customerID` | Keep it for now | It is useful for tracing rows during cleaning. It will be removed from model features later. |
| Duplicate rows/customer IDs | Validate and report | We do not silently remove records unless we find a real issue. |

## 1. Setup

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "raw" / "Telco-Customer-Churn.csv").exists():
            return path
    raise FileNotFoundError("Could not find project root with data/raw/Telco-Customer-Churn.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "telco_churn_cleaned.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Interim data path: {INTERIM_DATA_PATH}")

Project root: /home/patri/master/2026-06-ads-II-master
Raw data path: /home/patri/master/2026-06-ads-II-master/data/raw/Telco-Customer-Churn.csv
Interim data path: /home/patri/master/2026-06-ads-II-master/data/interim/telco_churn_cleaned.csv


Un proyecto profesional no usa rutas escritas a mano. El código debe encontrar automáticamente la raíz del proyecto y trabajar con rutas relativas.

## 2. Load Raw Data

In [3]:
def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Raw data file not found: {path}")

    df = pd.read_csv(path)
    print(f"Loaded raw data with {df.shape[0]:,} rows and {df.shape[1]:,} columns.")
    return df


raw_df = load_raw_data(RAW_DATA_PATH)
raw_df.head()

Loaded raw data with 7,043 rows and 21 columns.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Inspect Known Quality Issues

In [4]:
total_charges_numeric = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")
blank_total_charges = raw_df[total_charges_numeric.isna()]

print(f"Rows with blank/non-numeric TotalCharges: {len(blank_total_charges)}")
blank_total_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Rows with blank/non-numeric TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


Data Validation before Cleaning

Antes de modificar nada, comprobamos que nuestras hipótesis sobre los datos son ciertas.

Chequeamos que efectivamente hay totalcharges como NaN para gestionarlo mas tarde

In [5]:
print(f"Duplicate rows: {raw_df.duplicated().sum()}")
print(f"Duplicate customerID values: {raw_df['customerID'].duplicated().sum()}")
print(f"Churn values: {sorted(raw_df['Churn'].dropna().unique())}")

Duplicate rows: 0
Duplicate customerID values: 0
Churn values: ['No', 'Yes']


Está comprobando que la variable objetivo contiene únicamente los valores esperados.

## 4. Define Cleaning Function

In [6]:
def clean_telco_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the raw Telco Customer Churn dataset.

    The function is intentionally compact because this is still the notebook phase.
    It keeps row-level traceability by preserving customerID.
    """
    cleaned = df.copy()

    cleaned.columns = cleaned.columns.str.strip()

    required_columns = {"customerID", "TotalCharges", "tenure", "Churn"}
    missing_columns = required_columns - set(cleaned.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    if cleaned["customerID"].duplicated().any():
        raise ValueError("Duplicate customerID values found. Investigate before continuing.")

    churn_mapping = {"No": 0, "Yes": 1}
    unexpected_churn_values = set(cleaned["Churn"].dropna().unique()) - set(churn_mapping)
    if unexpected_churn_values:
        raise ValueError(f"Unexpected Churn values: {sorted(unexpected_churn_values)}")

    cleaned["TotalCharges"] = pd.to_numeric(cleaned["TotalCharges"], errors="coerce")

    missing_total_charges = cleaned["TotalCharges"].isna()
    missing_with_nonzero_tenure = missing_total_charges & (cleaned["tenure"] != 0)
    if missing_with_nonzero_tenure.any():
        problem_rows = cleaned.loc[
            missing_with_nonzero_tenure,
            ["customerID", "tenure", "MonthlyCharges", "TotalCharges"],
        ]
        raise ValueError(
            "Found missing TotalCharges for customers with tenure > 0. "
            f"Problem rows: {problem_rows.to_dict(orient='records')}"
        )

    cleaned.loc[missing_total_charges, "TotalCharges"] = 0.0
    cleaned["ChurnBinary"] = cleaned["Churn"].map(churn_mapping).astype("int64")

    return cleaned


cleaned_df = clean_telco_data(raw_df)
cleaned_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBinary
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


copiar datos
↓
validar estructura
↓
validar IDs
↓
validar target
↓
convertir TotalCharges
↓
validar regla de negocio
↓
rellenar casos válidos
↓
crear target numérico
↓
devolver dataset limpio

No limpiamos a ciegas. Validamos primero, limpiamos después.


## 5. Validate Cleaned Data

In [7]:
print(f"Raw shape:     {raw_df.shape}")
print(f"Cleaned shape: {cleaned_df.shape}")

Raw shape:     (7043, 21)
Cleaned shape: (7043, 22)


In [8]:
validation_summary = pd.DataFrame({
    "missing_count": cleaned_df.isna().sum(),
    "dtype": cleaned_df.dtypes.astype(str),
    "unique_values": cleaned_df.nunique(),
})

validation_summary

,missing_count,dtype,unique_values
customerID,0,object,7043
gender,0,object,2
SeniorCitizen,0,int64,2
Partner,0,object,2
Dependents,0,object,2
tenure,0,int64,73
PhoneService,0,object,2
MultipleLines,0,object,3
InternetService,0,object,3
OnlineSecurity,0,object,3


In [9]:
cleaned_df.loc[
    raw_df["TotalCharges"].astype(str).str.strip().eq(""),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn", "ChurnBinary"],
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn,ChurnBinary
488,4472-LVYGI,0,52.55,0.0,No,0
753,3115-CZMZD,0,20.25,0.0,No,0
936,5709-LVOEQ,0,80.85,0.0,No,0
1082,4367-NUYAO,0,25.75,0.0,No,0
1340,1371-DWPAZ,0,56.05,0.0,No,0
3331,7644-OMVMY,0,19.85,0.0,No,0
3826,3213-VVOLG,0,25.35,0.0,No,0
4380,2520-SGTTA,0,20.00,0.0,No,0
5218,2923-ARZLG,0,19.70,0.0,No,0
6670,4075-WKNIU,0,73.35,0.0,No,0


In [10]:
pd.DataFrame({
    "label": ["No", "Yes"],
    "encoded_value": [0, 1],
    "count": [
        (cleaned_df["Churn"] == "No").sum(),
        (cleaned_df["Churn"] == "Yes").sum(),
    ],
})

,label,encoded_value,count
0,No,0,5174
1,Yes,1,1869


## 6. Save Interim Dataset

In [11]:
INTERIM_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(INTERIM_DATA_PATH, index=False)

print(f"Saved cleaned interim data to: {INTERIM_DATA_PATH}")

Saved cleaned interim data to: /home/patri/master/2026-06-ads-II-master/data/interim/telco_churn_cleaned.csv


In [12]:
reloaded_df = pd.read_csv(INTERIM_DATA_PATH)

print(f"Reloaded shape: {reloaded_df.shape}")
reloaded_df.head()

Reloaded shape: (7043, 22)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBinary
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


## Cleaning Summary

The cleaned interim dataset is now ready for feature engineering.

What changed:

- `TotalCharges` is numeric.
- Blank `TotalCharges` values for customers with `tenure == 0` were set to `0.0`.
- `ChurnBinary` was added as a numeric target column.
- `customerID` is still present for traceability.
- No rows were dropped.

Next notebook step: feature engineering.